# Complete Pipeline: Fine-Tuning, Inference, ONNX and Quantization

This notebook contains the workflow to fine-tune a BERT model, perform live inference through a chat interface, export to ONNX format, and apply dynamic INT8 quantization.

### 1. Dependency Installation
Install the necessary libraries for model training, ONNX export, and optimization.

In [3]:
# Install required libraries
!pip install -q torch transformers datasets accelerate onnx onnxruntime optimum[onnxruntime]

### 2. Dataset Preparation
Load the dataset directly from a local JSONL file for training.

In [7]:

from datasets import load_dataset

# Direct load of the JSONL file. This creates the Dataset object internally.
dataset = load_dataset("json", data_files="papper_data.jsonl", split="train")

### 3. Tokenization and Model Configuration
Initialize the tokenizer and BERT sequence classification model, then apply tokenization.

In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Configure GPU usage with CUDA
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Function to process and tokenize the text
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 1323/1323 [00:00<00:00, 14172.77 examples/s]


### 4. Fine-Tuning Process
Set up the training parameters and start the fine-tuning process using the Trainer API.

In [9]:
from transformers import TrainingArguments, Trainer

# Training parameters using fp16 mixed precision for faster training
training_args = TrainingArguments(
    output_dir="./resultados",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=2,
    fp16=True,                      # Enables mixed precision
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()

# Save the locally trained base model
model.save_pretrained("./output")
tokenizer.save_pretrained("./output")
print("Model saved in './output'")

Step,Training Loss
2,0.660300
4,0.618100
6,0.569700
8,0.472300
10,0.467600
12,0.456900
14,0.294300
16,0.217900
18,0.191400
20,0.108300


Modelo guardado en './output'


### 5. Live Inference Interface
An interactive loop to test the trained model's predictions in real-time.

In [10]:
print("Interactive chat started. Type 'exit' to terminate.\n")
model.eval()

while True:
    texto = input("Enter text to classify: ")
    if texto.lower() == 'exit':
        print("Interface terminated.")
        break
        
    inputs = tokenizer(texto, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        prediccion = torch.argmax(outputs.logits, dim=1).item()
        
    clase = "Filtro / Documentación (1)" if prediccion == 1 else "Informativo / General (0)"
    print(f"Prediction: {clase}\n")

Chat interactivo iniciado. Escribe 'salir' para terminar.

Predicción: Filtro / Documentación (1)

Predicción: Informativo / General (0)

Predicción: Informativo / General (0)

Predicción: Filtro / Documentación (1)

Predicción: Filtro / Documentación (1)

Predicción: Informativo / General (0)



KeyboardInterrupt: Interrupted by user

### 6. Export to ONNX Format
Convert the PyTorch model to ONNX format to optimize inference speed.

In [12]:
from optimum.onnxruntime import ORTModelForSequenceClassification

# Export PyTorch model to ONNX format optimizing for deployment
onnx_path = "./modelo_onnx"
ort_model = ORTModelForSequenceClassification.from_pretrained("./output", export=True)
ort_model.save_pretrained(onnx_path)
tokenizer.save_pretrained(onnx_path)
print(f"Model successfully exported to ONNX in: {onnx_path}")

`torch_dtype` is deprecated! Use `dtype` instead!
/home/jrodriiguezg/.local/lib/python3.14/site-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


Modelo exportado correctamente a ONNX en: ./modelo_onnx


### 7. Dynamic INT8 Quantization of the ONNX Model
Apply quantization to reduce the model size and accelerate CPU inference.

In [13]:
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

# Initialize the quantizer with the newly generated ONNX file
quantizer = ORTQuantizer.from_pretrained(onnx_path, file_name="model.onnx")

# Configure dynamic quantization optimized for x86 architectures (AVX2/AVX512)
qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)

# Execute quantization and store the results
quantizer.quantize(save_dir="./modelo_onnx_cuantizado", quantization_config=qconfig)
print("INT8 Quantization completed successfully in './modelo_onnx_cuantizado'")

Cuantización INT8 completada con éxito en './modelo_onnx_cuantizado'


### 8. Verification of the Quantized Model
Test the final ONNX quantized model to ensure inference works as expected.

In [14]:
from transformers import AutoTokenizer
# Verify that the final optimized model performs inference correctly using ONNX Runtime
tokenizer_q = AutoTokenizer.from_pretrained("./modelo_onnx_cuantizado")
model_q = ORTModelForSequenceClassification.from_pretrained("./modelo_onnx_cuantizado", file_name="model_quantized.onnx")

texto_prueba = "necesito recuperar el resguardo del pago"
inputs_q = tokenizer_q(texto_prueba, return_tensors="pt")
outputs_q = model_q(**inputs_q)

print(f"Texto: '{texto_prueba}'")
print("Quantized model logits:", outputs_q.logits)

Texto: 'necesito recuperar el resguardo del pago'
Logits del modelo cuantizado: tensor([[-4.4219,  4.4207]])
